# Lab 04: Eigenvalues, SVD & PCA
> วิชา 1145 201 คณิตศาสตร์สำหรับวิทยาการข้อมูล | Week 4 | CLO1 | Strang Ch.6–7

---

สัปดาห์นี้เราจะสำรวจ 3 แนวคิดที่เชื่อมโยงกันอย่างลึกซึ้ง ได้แก่ **Eigenvalues/Eigenvectors**, **Singular Value Decomposition (SVD)** และ **Principal Component Analysis (PCA)** ซึ่งเป็นเครื่องมือ Linear Algebra ที่นักวิทยาการข้อมูลใช้บ่อยที่สุด Eigenvalue λ บอกว่า transformation ยืด/หดข้อมูลในแต่ละทิศทางเท่าไร โดยไม่เปลี่ยนทิศทางของ eigenvector **x** SVD แยก matrix ใดๆ (ไม่ต้องเป็น square) ออกเป็น A = UΣVᵀ ซึ่งทำให้หา low-rank approximation ที่ดีที่สุดของ matrix ได้ PCA ใช้หลักการนี้ในการหาทิศทางที่ข้อมูลกระจายตัวมากที่สุดเพื่อลด dimension โดยสูญเสีย information น้อยที่สุด เป้าหมายของ lab นี้คือให้นักศึกษาคำนวณ eigenvalues/eigenvectors ด้วย NumPy ได้ อธิบาย SVD และ reconstruct matrix ได้ และทำ PCA ทั้งแบบ from scratch และแบบ sklearn ได้อย่างถูกต้อง ในชีวิตจริง PCA ถูกใช้ใน image compression, face recognition (Eigenfaces), genomics และเป็น preprocessing step ก่อนสร้าง ML model เพื่อลด overfitting

**LLo:** คำนวณ Eigenvalue/Eigenvector, SVD และประยุกต์ใช้ PCA ลด dimension ชุดข้อมูลจริงได้

**สิ่งที่จะเรียนรู้ใน Lab นี้:**
- `np.linalg.eig` / `np.linalg.eigh` — eigendecomposition สำหรับ general และ symmetric matrix
- `np.linalg.svd` — full & reduced SVD และ low-rank approximation
- PCA from scratch: center → covariance → eigenvectors → project
- `sklearn.decomposition.PCA` เปรียบเทียบกับ from-scratch
- Scree Plot และ explained variance ratio
- **Case Study**: Low-rank approximation บน digit images (SVD image compression)

In [ ]:
# ─── นำเข้า library ทั้งหมดที่ใช้ใน Lab 04 ────────────────────────────────
# วัตถุประสงค์: เตรียม tools สำหรับ numerical computation, ML, และ visualization
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits
from sklearn.decomposition import PCA as SklearnPCA

# ─── ตั้งค่า random seed และ plot style ────────────────────────────────────
# วัตถุประสงค์: ให้ผลลัพธ์ reproducible และ plot อ่านง่าย
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully')
print(f'NumPy version: {np.__version__}')

## Part 1: Eigenvalues & Eigenvectors

**Part นี้เราจะทำอะไร:** คำนวณ Eigenvalues และ Eigenvectors ของ matrix ด้วย NumPy  
**เพื่ออะไร:** เพื่อเข้าใจว่า linear transformation ยืด/หดข้อมูลในแต่ละทิศทางอย่างไร ซึ่งเป็นพื้นฐานของ PCA และ SVD  
**ขั้นตอนที่จะทำ:**
1. สร้าง matrix A คำนวณ eigenvalues/eigenvectors ด้วย `np.linalg.eig`
2. ตรวจสอบสมการ **Ax = λx** ว่าเป็นจริงหรือไม่
3. เปรียบเทียบ `np.linalg.eig` (general) กับ `np.linalg.eigh` (symmetric)
4. ทำ TODO 1: คำนวณ eigenvalues ของ covariance matrix

### ทบทวนสูตร
ถ้า **A** คือ matrix n×n, eigenvector **x** และ eigenvalue λ ตอบสนอง:

```
Ax = λx
```

ความหมาย: **A** transform **x** โดยยืด/หดด้วยปัจจัย λ โดยไม่เปลี่ยนทิศทาง  
- Trace(A) = Σλᵢ (sum of eigenvalues)
- Det(A) = Πλᵢ (product of eigenvalues)

In [ ]:
# ─── สาธิต Eigenvalues/Eigenvectors บน matrix 2×2 ────────────────────────
# วัตถุประสงค์: แสดงให้เห็นว่า np.linalg.eig ทำงานอย่างไร
#              และตรวจสอบ Ax = lambda*x เป็นจริงหรือไม่

A = np.array([[3, 1],
              [0, 2]], dtype=float)

print('Matrix A:')
print(A)
print(f'\nTrace(A) = {np.trace(A):.2f}  (ควรเท่ากับ sum of eigenvalues)')
print(f'Det(A)   = {np.linalg.det(A):.2f}  (ควรเท่ากับ product of eigenvalues)')

# ─── คำนวณ eigenvalues และ eigenvectors ────────────────────────────────────
# วัตถุประสงค์: eigenvalues อยู่ใน vals, eigenvectors เป็น columns ของ vecs
vals, vecs = np.linalg.eig(A)

print(f'\nEigenvalues λ: {vals}')
print(f'Eigenvector matrix (columns = eigenvectors):\n{np.round(vecs, 4)}')

# ─── ตรวจสอบ Ax = lambda * x ──────────────────────────────────────────────
# วัตถุประสงค์: verify ว่า eigenvalue equation เป็นจริงด้วยตัวเลข
print('\n--- ตรวจสอบ Ax = λx ---')
for i in range(len(vals)):
    x = vecs[:, i]
    lam = vals[i]
    residual = np.abs(A @ x - lam * x).max()
    print(f'  λ={lam:.1f}: ||Ax - λx||_max = {residual:.2e}  (≈ 0 = ถูกต้อง)')

print(f'\nVerify Trace = sum(λ): {np.trace(A):.2f} = {vals.sum():.2f}')
print(f'Verify Det = prod(λ): {np.linalg.det(A):.2f} = {vals.prod():.2f}')

In [ ]:
# ─── เปรียบเทียบ np.linalg.eig กับ np.linalg.eigh ────────────────────────
# วัตถุประสงค์: eigh ออกแบบสำหรับ symmetric matrix — ให้ real eigenvalues
#              เรียงจากน้อยไปมาก และ eigenvectors orthonormal
#              ใช้ eigh เสมอสำหรับ covariance matrix ใน PCA

S = np.array([[4, 2],
              [2, 3]], dtype=float)

print('Symmetric matrix S (S = Sᵀ):')
print(S)
print(f'Is symmetric: {np.allclose(S, S.T)}')

# eig — general algorithm (สำหรับ matrix ทั่วไป)
vals_eig, vecs_eig = np.linalg.eig(S)

# eigh — optimized สำหรับ symmetric/Hermitian matrix
vals_eigh, vecs_eigh = np.linalg.eigh(S)

print(f'\nnp.linalg.eig  → λ: {vals_eig.round(4)}  (อาจไม่เรียง)')
print(f'np.linalg.eigh → λ: {vals_eigh.round(4)}  (sorted ascending)')

# ─── ตรวจสอบ orthonormality ของ eigenvectors จาก eigh ──────────────────────
# วัตถุประสงค์: VᵀV = I แสดงว่า eigenvectors orthonormal (spectral theorem)
VtV = vecs_eigh.T @ vecs_eigh
print(f'\nVᵀV (eigh) — ควรเป็น Identity matrix:')
print(np.round(VtV, 10))
print('\nข้อสรุป: สำหรับ covariance matrix (symmetric) ใช้ eigh เสมอ')

In [ ]:
# ── TODO 1 (Easy): Eigenvalues ของ Covariance Matrix ────────────────────────
#
# บริบท: Covariance matrix เป็น symmetric positive semi-definite
#        eigenvalues ของมันคือ variance ตามแต่ละทิศทาง principal component
#        นี่คือหัวใจของ PCA ที่เราจะใช้ใน Part 3
#
# โจทย์: กำหนดข้อมูล X_data shape (100, 3)
#   1. Center X_data (ลบ column mean ออกจากแต่ละ column)
#   2. คำนวณ covariance matrix: C = (1/(n-1)) * X_centered.T @ X_centered
#   3. หา eigenvalues ด้วย np.linalg.eigh (เหมาะกับ symmetric matrix)
#   4. print eigenvalues (ascending) และ total variance
#   5. บอกว่า eigenvector ตัวไหนอธิบาย variance ได้มากที่สุด
#
# ผลลัพธ์ที่คาดหวัง: eigenvalues 3 ค่า, eigenvalue ที่ใหญ่สุด ≈ > 3.0

np.random.seed(42)
X_data = np.random.randn(100, 3) @ np.array([[2.0, 0.5, 0.0],
                                               [0.5, 1.0, 0.3],
                                               [0.0, 0.3, 0.5]])
print('X_data shape:', X_data.shape)
print('First 3 rows:')
print(X_data[:3].round(3))
print()

# --- เขียน code ของคุณที่นี่ ---
# n = X_data.shape[0]
# X_centered = ...
# C = ...
# eigenvalues, eigenvectors = ...
# print('Eigenvalues (ascending):', eigenvalues.round(4))
# print('Total variance (sum of eigenvalues):', eigenvalues.sum().round(4))
# print('Largest eigenvalue corresponds to PC with most variance')

## Part 2: Singular Value Decomposition (SVD)

**Part นี้เราจะทำอะไร:** คำนวณ SVD ของ matrix และ reconstruct matrix จาก U, Σ, Vᵀ  
**เพื่ออะไร:** SVD ทำงานกับ matrix ทุกรูปร่าง (ไม่ต้องเป็น square) ทำให้ใช้กับ data matrix จริงได้เสมอ และเป็น foundation ทางคณิตศาสตร์ของ PCA  
**ขั้นตอนที่จะทำ:**
1. SVD ของ matrix 4×3 และ reconstruct A = UΣVᵀ
2. เข้าใจความแตกต่างระหว่าง full SVD และ reduced SVD
3. ทำ TODO 2: reduced SVD และ low-rank approximation

### สูตร SVD: A = UΣVᵀ
```
A (m×n)  =  U (m×m)  ×  Σ (m×n)  ×  Vᵀ (n×n)
             orthogonal    diagonal    orthogonal
             left SV       sing. val   right SV transposed
```
**สำคัญ:** `np.linalg.svd` คืน **Vᵀ** (V transposed) ไม่ใช่ V ต้องระวังเวลา reconstruct

In [ ]:
# ─── Full SVD บน matrix 4×3 ────────────────────────────────────────────────
# วัตถุประสงค์: แสดงขนาดของ U, s, Vt และ verify A = U @ diag(s) @ Vt

A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9],
              [1, 3, 2]], dtype=float)

print(f'Matrix A shape: {A.shape}  (4 rows × 3 cols)')
print('A:')
print(A)

# ─── คำนวณ Full SVD ───────────────────────────────────────────────────────
# วัตถุประสงค์: full_matrices=True = full SVD: U(4×4), Vt(3×3)
U, s, Vt = np.linalg.svd(A, full_matrices=True)

print(f'\nU shape: {U.shape}  (left singular vectors)')
print(f's shape: {s.shape}  (singular values — 1D array, min(m,n) values)')
print(f'Vt shape: {Vt.shape}  (Vᵀ — right singular vectors transposed)')
print(f'\nSingular values σ: {s.round(4)}')
print(f'Note: σ₁ >> σ₂ >> σ₃ ≈ 0 → matrix มี numerical rank ≈ 2')

# ─── Reconstruct A = U @ Sigma @ Vt ──────────────────────────────────────
# วัตถุประสงค์: ต้องสร้าง Sigma matrix (4×3) จาก vector s ก่อน
Sigma = np.zeros(A.shape)        # zero matrix ขนาดเดียวกับ A (4×3)
np.fill_diagonal(Sigma, s)       # ใส่ singular values บน diagonal

A_reconstructed = U @ Sigma @ Vt
error = np.linalg.norm(A - A_reconstructed, 'fro')
print(f'\nReconstruction error (Frobenius norm): {error:.2e}  (ควร ≈ 0)')

In [ ]:
# ── TODO 2 (Medium): Reduced SVD & Low-rank Approximation ──────────────────
#
# บริบท: ในทางปฏิบัติ เราใช้ reduced SVD (เก็บเฉพาะ r singular values แรก)
#        เพราะ singular values ขนาดเล็กส่วนใหญ่เป็น noise
#        Low-rank approximation: A_k = Σᵢ₌₁ᵏ σᵢ × uᵢ × vᵢᵀ
#        Eckart-Young Theorem: A_k คือ best rank-k approximation ใน Frobenius norm
#
# โจทย์: ใช้ matrix A จาก cell ก่อน
#   1. คำนวณ Reduced SVD ด้วย full_matrices=False
#      → U_r(4×3), s_r(3,), Vt_r(3×3)
#   2. สร้าง rank-1 approximation: A_k1 = s_r[0] * outer(U_r[:,0], Vt_r[0,:])
#   3. สร้าง rank-2 approximation: A_k2 (ใช้ 2 singular values แรก)
#   4. คำนวณ Frobenius norm error สำหรับ k=1 และ k=2
#   5. คำนวณ % variance explained: sum(s[:k]**2) / sum(s**2) * 100
#
# ผลลัพธ์ที่คาดหวัง: k=2 ควร explain ≈ 99%+ ของ variance

# ─── Reduced SVD ────────────────────────────────────────────────────────
# วัตถุประสงค์: full_matrices=False ประหยัด memory, U_r(m×r), Vt_r(r×n)
U_r, s_r, Vt_r = np.linalg.svd(A, full_matrices=False)
print(f'Reduced SVD: U_r{U_r.shape}, s_r{s_r.shape}, Vt_r{Vt_r.shape}')

# --- เขียน code ของคุณที่นี่ ---
# rank-1 approximation
# A_k1 = s_r[0] * np.outer(U_r[:, 0], Vt_r[0, :])
#
# rank-2 approximation
# A_k2 = ...
#
# Frobenius norm errors
# error_k1 = ...
# error_k2 = ...
#
# % variance explained
# var_k1 = ...
# var_k2 = ...
#
# print(f'k=1: error={error_k1:.3f}, variance explained={var_k1:.1f}%')
# print(f'k=2: error={error_k2:.3f}, variance explained={var_k2:.1f}%')

## Part 3: PCA from Scratch

**Part นี้เราจะทำอะไร:** ทำ PCA บน Iris dataset โดย implement ทุกขั้นตอนด้วยตัวเอง  
**เพื่ออะไร:** เพื่อเข้าใจว่า PCA ทำงานอย่างไรจาก first principles — ก่อนใช้ sklearn  
**ขั้นตอน PCA from Scratch:**
```
1. Center data     : X_centered = X - mean(X, axis=0)
2. Covariance      : C = (1/(n-1)) * X_centered.T @ X_centered
3. Eigen-decompose : C = Q Λ Qᵀ  (ใช้ np.linalg.eigh)
4. Sort descending : เรียง eigenvectors ตาม eigenvalue มากไปน้อย
5. Project         : X_pca = X_centered @ Q_k  (k คือ จำนวน PCs ที่ต้องการ)
```

In [ ]:
# ─── โหลด Iris dataset ────────────────────────────────────────────────────
# วัตถุประสงค์: Iris มี 4 features (sepal/petal) เราจะลดเหลือ 2 ด้วย PCA
iris = load_iris()
X = iris.data        # shape (150, 4)
y = iris.target      # 0=setosa, 1=versicolor, 2=virginica

print(f'X shape: {X.shape}  (150 samples, 4 features)')
print(f'Features: {iris.feature_names}')

# ─── Step 1: Center data ──────────────────────────────────────────────────
# วัตถุประสงค์: PCA ต้องการ zero-mean data ไม่งั้น covariance matrix จะ biased
X_mean = X.mean(axis=0)
X_centered = X - X_mean
print(f'\nColumn means before centering: {X_mean.round(3)}')
print(f'Column means after centering:  {X_centered.mean(axis=0).round(10)}')

# ─── Step 2: Covariance matrix ────────────────────────────────────────────
# วัตถุประสงค์: C[i,j] = covariance ระหว่าง feature i กับ feature j
#              หาร (n-1) เพื่อ unbiased estimate (Bessel's correction)
n = X_centered.shape[0]
C = (1 / (n - 1)) * X_centered.T @ X_centered  # (4×4) symmetric matrix
print(f'\nCovariance matrix C (4×4):')
print(C.round(4))

# ─── Step 3: Eigen-decompose C ────────────────────────────────────────────
# วัตถุประสงค์: eigh คืน eigenvalues ascending → ต้อง reverse เพื่อ descending
eigenvalues, eigenvectors = np.linalg.eigh(C)

# ─── Step 4: Sort descending ──────────────────────────────────────────────
# วัตถุประสงค์: PC1 ต้องมี eigenvalue มากที่สุด (explain variance มากที่สุด)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print(f'\nEigenvalues (descending): {eigenvalues.round(4)}')
print(f'Total variance = Σλᵢ = {eigenvalues.sum():.4f} = Trace(C) = {np.trace(C):.4f}')

# ─── Step 5: Project onto top 2 PCs ──────────────────────────────────────
# วัตถุประสงค์: X_pca = พิกัดของแต่ละ sample ใน 2D PCA space
Q2 = eigenvectors[:, :2]          # top-2 eigenvectors: shape (4, 2)
X_pca = X_centered @ Q2           # (150, 4) @ (4, 2) = (150, 2)
print(f'\nX_pca shape: {X_pca.shape}  (150 samples × 2 PCs)')

In [ ]:
# ── TODO 3 (Medium): Explained Variance Ratio ──────────────────────────────
#
# บริบท: เราต้องรู้ว่า top-k PCs อธิบาย variance ได้กี่ % ของ total variance
#        ถ้า PC1+PC2 อธิบายได้ >= 95% แปลว่าการลดเหลือ 2D แทบไม่สูญเสีย information
#        ใช้ criterion นี้ในการเลือก k ที่เหมาะสม
#
# โจทย์: ใช้ eigenvalues ที่คำนวณไว้แล้วใน cell ก่อน
#   1. คำนวณ explained_ratio: eigenvalue[i] / sum(eigenvalues) สำหรับแต่ละ i
#   2. คำนวณ cumulative_ratio: np.cumsum(explained_ratio)
#   3. print ตาราง: PC | Eigenvalue | Explained% | Cumulative%
#   4. หา k_95: จำนวน PC น้อยสุดที่ cumulative_ratio >= 95%
#
# ผลลัพธ์ที่คาดหวัง: PC1+PC2 ควร explain ≈ 97%+ ของ Iris data

# --- เขียน code ของคุณที่นี่ ---
# explained_ratio = eigenvalues / eigenvalues.sum()
# cumulative_ratio = np.cumsum(explained_ratio)
#
# print(f"{'PC':<5} {'Eigenvalue':>12} {'Explained%':>12} {'Cumulative%':>12}")
# print('-' * 45)
# for i, (ev, er, cr) in enumerate(zip(eigenvalues, explained_ratio, cumulative_ratio)):
#     print(f"PC{i+1:<4} {ev:>12.4f} {er*100:>11.2f}% {cr*100:>11.2f}%")
#
# k_95 = np.argmax(cumulative_ratio >= 0.95) + 1
# print(f'\nต้องใช้ {k_95} PC เพื่อให้ได้ >= 95% variance')

## Part 4: PCA ด้วย sklearn + Scree Plot

**Part นี้เราจะทำอะไร:** ทำ PCA ด้วย `sklearn.decomposition.PCA` และสร้าง Scree Plot  
**เพื่ออะไร:** sklearn PCA ใช้ SVD internally ซึ่ง numerically stable กว่า eigendecomposition สำหรับ large dataset และ API ใช้งานง่ายกว่า  
**ขั้นตอนที่จะทำ:**
1. fit sklearn PCA บน Iris data
2. เปรียบเทียบผลกับ from-scratch
3. TODO 4: สร้าง Scree Plot และ 2D scatter plot

In [ ]:
# ─── PCA ด้วย sklearn ──────────────────────────────────────────────────────
# วัตถุประสงค์: sklearn PCA centering อัตโนมัติ, ใช้ SVD internally
pca = SklearnPCA(n_components=4)     # เก็บทุก component เพื่อดู scree plot
X_pca_sk = pca.fit_transform(X)      # center + project ใน 1 step

print('sklearn PCA — explained_variance_ratio_:')
for i, ratio in enumerate(pca.explained_variance_ratio_):
    cumul = pca.explained_variance_ratio_[:i+1].sum()
    print(f'  PC{i+1}: {ratio*100:6.2f}%   (cumulative: {cumul*100:.2f}%)')

# ─── เปรียบเทียบกับ from-scratch ──────────────────────────────────────────
# วัตถุประสงค์: ตรวจสอบว่าผลสอดคล้องกัน
#              อาจต่างแค่เครื่องหมาย (eigenvector direction ไม่ unique)
print('\n--- เปรียบเทียบ PC1 coordinates (5 samples แรก) ---')
print(f'from-scratch: {X_pca[:5, 0].round(4)}')
print(f'sklearn:      {X_pca_sk[:5, 0].round(4)}')
print('(อาจต่างแค่เครื่องหมาย — นั่นคือ principal directions เดียวกัน)')

In [ ]:
# ── TODO 4 (Hard): Scree Plot + 2D PCA Scatter ─────────────────────────────
#
# บริบท: Scree Plot แสดง explained variance % ต่อ PC ให้เห็น 'elbow point'
#        ซึ่งช่วยตัดสินใจว่าควรเลือก k PCs เท่าไร
#        2D scatter แสดงว่า PCA สามารถแยกกลุ่ม species ออกจากกันได้ดีแค่ไหน
#
# โจทย์: สร้าง subplot 1×2 (figsize=(14, 5))
#
# Subplot ซ้าย — Scree Plot:
#   - bar chart: x=PC1..PC4, y=explained variance % (จาก pca.explained_variance_ratio_)
#   - Twin y-axis: line plot cumulative variance %, horizontal dashed line ที่ 95%
#   - Title: 'Scree Plot — Iris Dataset'
#   - Labels: xlabel='Principal Component', ylabel (left)='Explained Variance %',
#             ylabel (right)='Cumulative %'
#
# Subplot ขวา — 2D Scatter (PC1 vs PC2):
#   - scatter แต่ละ species ด้วยสีต่างกัน
#   - xlabel: f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)'
#   - ylabel: f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'
#   - legend แสดง species names (iris.target_names)
#   - Title: '2D PCA Projection — Iris Dataset'

# --- เขียน code ของคุณที่นี่ ---
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
#
# colors = ['#e74c3c', '#2ecc71', '#3498db']
# pcs = [f'PC{i+1}' for i in range(4)]
# var_pct = pca.explained_variance_ratio_ * 100
#
# --- Scree Plot ---
# ax1.bar(...)
# ax1_twin = ax1.twinx()
# ax1_twin.plot(...)
# ax1_twin.axhline(y=95, ...)
#
# --- 2D Scatter ---
# for i, (name, color) in enumerate(zip(iris.target_names, colors)):
#     mask = y == i
#     ax2.scatter(X_pca_sk[mask, 0], X_pca_sk[mask, 1], ...)
#
# plt.tight_layout()
# plt.show()

## Part 5: Case Study — Low-rank Image Approximation

**Part นี้เราจะทำอะไร:** ใช้ SVD ทำ low-rank approximation บน digit images  
**เพื่ออะไร:** แสดงให้เห็น trade-off ระหว่าง compression ratio กับ image quality ซึ่งเป็น application จริงของ SVD

---

```
╔══════════════════════════════════════════════════════════════╗
║  CASE STUDY: Low-rank Image Approximation (SVD Compression)  ║
╠══════════════════════════════════════════════════════════════╣
║  1. Scenario  บริษัท ML ต้องบีบอัดภาพใบหน้า 8×8 pixels     ║
║               เพื่อลด storage และเพิ่มความเร็วประมวลผล      ║
║  2. Data      sklearn digits: 1,797 ภาพ ขนาด 8×8 grayscale  ║
║  3. Method    SVD: A_k = Σᵢ₌₁ᵏ σᵢ × uᵢ × vᵢᵀ              ║
║               (Eckart-Young: best rank-k approximation)      ║
║  4. Result    เห็นใน TODO 5 — หา k ที่ให้ >= 95% energy     ║
║  5. Insight   k_min << 8 แสดงว่า digit image มี rank ต่ำ    ║
║               → natural images มี inherent low-dimensional   ║
║               structure ที่ SVD/PCA สามารถ capture ได้       ║
╚══════════════════════════════════════════════════════════════╝
```

In [ ]:
# ─── โหลด digit image และทำ SVD ────────────────────────────────────────────
# วัตถุประสงค์: ใช้ digit '0' เป็น demo สำหรับ low-rank approximation
digits = load_digits()
img = digits.images[0]              # shape (8, 8) — ภาพตัวเลข '0'

print(f'Image shape: {img.shape}')
print(f'Pixel range: {img.min():.0f} – {img.max():.0f}')
print(f'True label: {digits.target[0]}')

# ─── SVD ของ image matrix ────────────────────────────────────────────────
# วัตถุประสงค์: แยก image เป็น singular components เพื่อ compress
U_img, s_img, Vt_img = np.linalg.svd(img, full_matrices=True)

print(f'\nSingular values σ: {s_img.round(2)}')
print(f'σ₁/σ₂ = {s_img[0]/s_img[1]:.1f}x — σ₁ dominant มาก (image rank ต่ำ)')

# ─── Reconstruct ด้วย k=1, 2, 4, 8 singular values ────────────────────────
# วัตถุประสงค์: เปรียบเทียบคุณภาพ image ที่ระดับ compression ต่างกัน
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

axes[0].imshow(img, cmap='gray', vmin=0, vmax=16)
axes[0].set_title('Original\n(64 values)', fontsize=10)
axes[0].axis('off')

for j, k in enumerate([1, 2, 4, 8]):
    # ─── สร้าง rank-k approximation ───────────────────────────────────────
    # วัตถุประสงค์: A_k = sum_{i=0}^{k-1} σᵢ × uᵢ × vᵢᵀ (outer product sum)
    A_k = sum(s_img[i] * np.outer(U_img[:, i], Vt_img[i, :]) for i in range(k))
    energy_pct = (s_img[:k]**2).sum() / (s_img**2).sum() * 100
    storage = k * (1 + 8 + 8)     # k × (1 σ + 8 u-values + 8 v-values)
    axes[j+1].imshow(A_k, cmap='gray', vmin=0, vmax=16)
    axes[j+1].set_title(f'k={k}\n{energy_pct:.0f}% energy\n({storage} values)', fontsize=9)
    axes[j+1].axis('off')

plt.suptitle('SVD Low-rank Approximation — Digit Image (8×8)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── TODO 5 (Hard): หา k_min ที่ capture >= 95% Energy ──────────────────────
#
# บริบท: ใน image compression เราต้องการ k ที่น้อยสุดที่ยังคงคุณภาพสูงพอ
#        95% energy threshold เป็น convention ที่ใช้บ่อย
#        นี่คือ practical guideline เดียวกับ PCA (95% cumulative variance)
#
# โจทย์: ใช้ s_img จาก cell ก่อน
#   1. คำนวณ cumulative energy: np.cumsum(s_img**2) / (s_img**2).sum()
#   2. หา k_min: np.argmax(cumulative_energy >= 0.95) + 1
#   3. Reconstruct image ด้วย k_min singular values
#   4. สร้าง figure 3 subplots:
#      - [ซ้าย]   Original image
#      - [กลาง]  k_min reconstruction พร้อม title บอก k_min และ % energy
#      - [ขวา]   Cumulative energy curve: x=k, y=cumulative energy %
#                เพิ่ม horizontal dashed line ที่ 95%
#                เพิ่ม vertical dashed line ที่ k_min
#   5. print compression ratio: 64 values → k_min × 17 values
#
# ผลลัพธ์ที่คาดหวัง: k_min = 3–4 สำหรับ 8×8 digit image

# --- เขียน code ของคุณที่นี่ ---
# cumulative_energy = ...
# k_min = ...
# A_kmin = ...
#
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# axes[0].imshow(img, ...)
# axes[1].imshow(A_kmin, ...)
# axes[2].plot(range(1, len(s_img)+1), cumulative_energy*100, ...)
# axes[2].axhline(y=95, ...)
# axes[2].axvline(x=k_min, ...)
#
# print(f'k_min = {k_min}')
# print(f'Compression: 64 values → {k_min * 17} values ({64/(k_min*17):.1f}x reduction)')

## สรุปและคำถาม Reflection

### สิ่งที่เรียนรู้ใน Lab นี้
- **`np.linalg.eigh`** เหมาะกับ symmetric matrix — คืน real eigenvalues เรียง ascending + orthonormal eigenvectors
- **SVD**: `np.linalg.svd(A, full_matrices=False)` คืน reduced SVD ประหยัด memory
- **PCA from scratch**: center → covariance → eigh → sort → project — เข้าใจทุกขั้นตอน
- **sklearn PCA** ใช้ SVD internally ซึ่ง stable กว่า eigendecomposition สำหรับ ill-conditioned matrix
- **Low-rank approximation**: rank-k SVD คือ best rank-k approximation (Eckart-Young theorem)

---

### คำถาม Reflection

**คำถาม 1:** จาก Scree Plot ของ Iris (TODO 4) — ถ้าเราใช้แค่ PC1 เดียว (1D)  
จะสูญเสีย information ไปกี่ % และในแง่ machine learning นั่นหมายความว่าอะไรสำหรับ classification task?

**คำถาม 2:** จาก TODO 5 คุณพบว่า k_min เท่าไรสำหรับ digit image 8×8?  
ลองอธิบายว่าทำไม digit image จึงมี effective rank ต่ำมาก (ทั้งที่มี 64 pixels)  
และ insight นี้บอกอะไรเกี่ยวกับ structure ของ natural images ทั่วไป?

---

*Lab 04 เสร็จสมบูรณ์ — ส่ง notebook นี้พร้อม output ของทุก cell (Run All Cells ก่อนส่ง)*